# 051 — Activaciones, inicialización y normalización

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** (a) Xavier: √(2/1024) = **0.0442**; He: √(2/512) = **0.0625**.
(b) Xavier: √(2/1280) = **0.0395**; He: √(2/1024) = **0.0442**. He solo mira n_in
porque el criterio se deriva de conservar la varianza en el forward con ReLU.

**Ejercicio 2.** μ = 5; σ² = (9+1+1+9)/4 = 5; √σ² = 2.2361.
x̂ = (−1.3416, −0.4472, +0.4472, +1.3416).
y = 0.5·x̂ − 1 = (−1.6708, −1.2236, −0.7764, −0.3292).
Media de y = −1 = β; desviación de y = 0.5 = γ. La normalización fija exactamente
los dos primeros momentos y deja que γ, β los reubiquen de forma aprendible.

**Ejercicio 3.** Sigmoide: 0.25⁸ = **1.5×10⁻⁵** — el gradiente llega a las primeras
capas prácticamente nulo. ReLU activa: 1⁸ = **1** — la magnitud se conserva. Esta es
la razón mecánica del cambio masivo a ReLU a partir de 2011-2012.

**Ejercicio 4.** Si w₁ = w₂ en t=0, ambas neuronas calculan la misma salida, reciben
el mismo δ en backprop y por tanto el mismo gradiente: w₁ = w₂ en todo t. La capa de
2 neuronas se comporta como una de 1. La inicialización aleatoria rompe la simetría.


In [ ]:
result = run_lab("neural", seed=51)
assert result["kind"] == "neural"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica
import math

# Ejercicio 1
print("Xavier 512→512:", round(math.sqrt(2/1024), 4), "| He:", round(math.sqrt(2/512), 4))
print("Xavier 1024→256:", round(math.sqrt(2/1280), 4), "| He:", round(math.sqrt(2/1024), 4))

# Ejercicio 2
x = [2, 4, 6, 8]; gamma, beta = 0.5, -1.0
mu = sum(x) / len(x)
var = sum((v - mu) ** 2 for v in x) / len(x)
x_hat = [(v - mu) / math.sqrt(var) for v in x]
y = [gamma * v + beta for v in x_hat]
media_y = sum(y) / len(y)
std_y = math.sqrt(sum((v - media_y) ** 2 for v in y) / len(y))
print("y =", [round(v, 4) for v in y], "| media:", round(media_y, 4), "| std:", round(std_y, 4))
assert abs(media_y - beta) < 1e-9 and abs(std_y - gamma) < 1e-9

# Ejercicio 3
print("factor sigmoide (8 capas):", 0.25 ** 8, "| factor ReLU:", 1 ** 8)


## Reflexión

1. ¿Por qué el factor de He es 2/n_in y no 1/n_in, y qué tiene que ver con la forma de ReLU?
2. Si una red con batch norm da métricas distintas entre entrenamiento y evaluación con los mismos datos, ¿cuál es la causa mecánica exacta?
3. ¿Qué observaciones (histogramas de activaciones, norma del gradiente por capa) te permitirían distinguir "gradiente que desaparece" de "neuronas ReLU muertas"?
